# Quadruped Locomotion (Go2): Reward Ablation

이 노트북은 `go2_locomotion_basic.ipynb` 이후에 실행되는 **reward ablation 실험용** 노트북입니다.

이전 노트북:
→ `go2_locomotion_basic.ipynb`

basic 노트북에서 다룬 **Task Reward만 사용하는 baseline(`Go2_first_test`)** 위에,
Reward 항을 단계적으로 추가하며 각 단계에서 어떤 행동 변화가 나타나는지 비교합니다.

| 실험 | 모델 | 변경 내용 |
|------|------|-----------|
| 실험 1 | `Go2_second_test` | 기본 Regularization + Gait Enforcement + Calf Early Termination 추가 |
| 실험 2 | `Go2_third_test` | Healthy Reward + Feet Air Time Reward 추가 |
| 실험 3 | `Go2_forth_test` | Hip Spread Penalty 추가 |

아래 셀부터 공통 세팅을 다시 수행합니다.

---

## 0. Environment Setup

Colab이면 `/content`를 기준 경로로 사용하고, 그렇지 않으면 현재 작업 디렉터리를 기준 경로로 사용합니다.

In [ ]:
# Clone repository
import os, sys

import yaml

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Cloned Directory already exists")

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "egl"

# numpy 1.x / 2.x 호환성 패치
import numpy, numpy.core, numpy.core.numeric, numpy.core.multiarray
import numpy.random._pickle as _np_pickle

sys.modules['numpy.core_'] = numpy.core
sys.modules['numpy.core_.numeric'] = numpy.core.numeric
sys.modules['numpy.core_.multiarray'] = numpy.core.multiarray

_orig_bg_ctor = _np_pickle.__bit_generator_ctor
def _patched_bg_ctor(bg='MT19937'):
    return bg() if isinstance(bg, type) else _orig_bg_ctor(bg)
_np_pickle.__bit_generator_ctor = _patched_bg_ctor

In [ ]:
# Install dependencies
# stable-baselines3는 PyPI에서 설치합니다 (이 repo에는 sb3 소스 포크가 없음).
# 로컬에서 검증된 버전 조합으로 고정합니다.
!pip install "stable-baselines3==2.3.0" "gymnasium==0.29.1" "mujoco==3.8.0" "numpy<2" "imageio[ffmpeg]" tensorboard pygments

In [ ]:
from pathlib import Path
from IPython.display import HTML
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, title="code", bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=title, max_height=max_height, bg=bg)

def show_func(obj, max_height=400, title="code", bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, title=title, max_height=max_height, bg=bg)

---

## 1. Reward Ablation Setup

학습 및 평가에 필요한 공통 함수와 변수를 선언합니다.

> Go2 환경은 항상 gait phase observation을 포함하므로,
> Go1 ablation notebook과 달리 with/without phase 분기가 필요하지 않습니다.

In [ ]:
import importlib
import numpy as np
import os
import gc
import time
import imageio
import torch
import shutil
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from IPython.display import Video, display
from pathlib import Path

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback

importlib.reload(go2_env)

policy_cfg_path = Path(repo_dir + "/src/params.yaml")
with policy_cfg_path.open("r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

# colab에서 실행하기 위한 설정
policy_cfg['n_envs'] = 12
policy_cfg['policy']['batch_size'] = 64

# 환경 확인
env = go2_env.Go2MujocoEnv(prj_path=repo_dir, render_mode=None)
obs, info = env.reset()
print(policy_cfg['n_envs'])
print(policy_cfg['policy']['batch_size'])
print(f"Observation shape: {np.array(obs).shape}")
print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")
env.close()


def train_run(env_cfg_path=None):
    importlib.reload(go2_env)

    MODEL_DIR = f"{repo_dir}/models"
    LOG_DIR = f"{repo_dir}/logs"

    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)

    vec_env = make_vec_env(
        go2_env.Go2MujocoEnv,
        env_kwargs={"prj_path": repo_dir, "cfg_path": env_cfg_path},
        n_envs=policy_cfg["n_envs"],
        seed=policy_cfg["seed"],
        vec_env_cls=SubprocVecEnv,
    )

    train_time = time.strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"{train_time}"
    model_path = f"{MODEL_DIR}/{run_name}"
    print(f"Training on {policy_cfg['n_envs']} envs, saving to '{model_path}'")

    # 사용한 envs.yaml 복사
    envs_src = Path(env_cfg_path) if env_cfg_path else Path(repo_dir) / "src" / "envs.yaml"
    envs_dst = Path(model_path) / "envs.yaml"
    envs_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(envs_src, envs_dst)
    print("Copied envs.yaml to:", envs_dst)

    checkpoint_callback = CheckpointCallback(
        save_freq=policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"],
        save_path=model_path,
        name_prefix="model",
        save_replay_buffer=False,
        save_vecnormalize=False,
    )
    eval_callback = EvalCallback(
        vec_env,
        best_model_save_path=model_path,
        log_path=LOG_DIR,
        eval_freq=policy_cfg["eval_freq"],
        n_eval_episodes=5,
        deterministic=True,
        render=False,
    )
    reward_logging_callback = RewardLoggingCallback()
    callbacks = CallbackList([eval_callback, checkpoint_callback, reward_logging_callback])

    model = PPO("MlpPolicy",
                env=vec_env,
                learning_rate=policy_cfg["policy"]["learning_rate"],
                n_steps=policy_cfg["policy"]["n_steps"],
                batch_size=policy_cfg["policy"]["batch_size"],
                n_epochs=policy_cfg["policy"]["n_epochs"],
                gamma=policy_cfg["policy"]["gamma"],
                gae_lambda=policy_cfg["policy"]["gae_lambda"],
                clip_range=policy_cfg["policy"]["clip_range"],
                normalize_advantage=policy_cfg["policy"]["normalize_advantage"],
                ent_coef=policy_cfg["policy"]["ent_coef"],
                vf_coef=policy_cfg["policy"]["vf_coef"],
                max_grad_norm=policy_cfg["policy"]["max_grad_norm"],
                verbose=1,
                tensorboard_log=LOG_DIR)

    model.learn(
        total_timesteps=policy_cfg["total_timestep"],
        reset_num_timesteps=True,
        progress_bar=True,
        tb_log_name=run_name,
        callback=callbacks,
    )
    model.save(f"{model_path}/final_model")
    vec_env.close()
    del model, eval_callback, vec_env
    gc.collect()


def eval_run():
    from tqdm.auto import tqdm

    importlib.reload(go2_env)
    model_path = f"{repo_dir}/models/{model_name}/best_model.zip"
    cfg_path = f"{repo_dir}/models/{model_name}/envs.yaml"
    print(f"Loading model from {model_path}")

    given_command = [0.9, 0.0, 0.0]
    env = go2_env.Go2MujocoEnv(
        prj_path=repo_dir,
        cfg_path=cfg_path,
        given_command=given_command,
        render_mode="rgb_array",
        camera_name="tracking",
        width=320,
        height=240,
    )
    env._reset_noise_scale = 0.05

    custom_objects = {
        "observation_space": env.observation_space,
        "action_space": env.action_space,
    }
    model = PPO.load(path=model_path, env=env, verbose=1, custom_objects=custom_objects)

    video_path = f"{repo_dir}/../rollout_{model_name}.mp4"

    obs, _ = env.reset()
    max_time_step_s = policy_cfg["test"]["max_time_step_s"]
    ep_len = 0
    t_render = 0.0
    n_render = 0
    last_render = 0.0
    start = time.perf_counter()
    video_fps = 10
    render_interval = 50 // video_fps
    max_steps = int(max_time_step_s * 50)

    frames = []
    pbar = tqdm(total=max_steps, desc="rollout", unit="step", dynamic_ncols=True)
    print("max time:", max_time_step_s, "  max steps:", max_steps)

    while ep_len < max_steps:
        with torch.no_grad():
            action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)

        if ep_len % render_interval == 0:
            t0 = time.perf_counter()
            frame = env.render()
            frames.append(frame)
            last_render = time.perf_counter() - t0
            t_render += last_render
            n_render += 1

        elapsed = time.perf_counter() - start
        steps_per_sec = ep_len / max(elapsed, 1e-9)
        avg_render = (t_render / n_render) if n_render else 0.0
        pbar.set_postfix({
            "steps/s": f"{steps_per_sec:6.1f}",
            "renders": n_render,
            "r_last(s)": f"{last_render:5.3f}",
            "r_avg(s)": f"{avg_render:5.3f}",
        })
        pbar.update(1)
        ep_len += 1

    imageio.mimwrite(video_path, frames, fps=video_fps, codec="libx264", quality=8, pixelformat="yuv420p")
    env.close()
    print("avg render sec:", t_render / max(n_render, 1))
    print("Saved video to:", video_path)
    return video_path

## 2. 실험 선택

진행할 실험 블록을 실행합니다.

각 실험에 사용된 reward 가중치는 모델 폴더 내 `envs.yaml`에 저장되어 있습니다.

새로운 reward를 추가하거나 가중치를 바꾸려면 `src/mdp/reward.py`, `src/go2_mujoco_env.py`의 `_get_reward()`, `src/envs.yaml`을 수정합니다.

### 실험 1 : 기본 Regularization + Gait Enforcement + Calf Early Termination 추가 (`Go2_second_test`)

- **추가 Reward**:
  - `torque`, `vertical_vel`, `xy_angular_vel`, `action_rate` — 모션 부드럽게
  - `joint_limit`, `joint_acc`, `action_norm`, `joint_pos_deviation` — 관절 안전 범위
  - `gait_enforcement` (0.05), `foot_clearance` (50) — trot gait 패턴 유도
- **Termination**: calf contact 시 조기 종료 추가 (`allow_calf_contact: false`)
- Task Reward만 있는 baseline(basic 노트북)보다 훨씬 안정적인 걸음걸이

In [ ]:
model_name = "Go2_second_test"
env_cfg_path = f"{repo_dir}/models/{model_name}/envs.yaml"

show_code(env_cfg_path, title=model_name + " envs.yaml", max_height=600)

### 실험 2 : Healthy Reward + Feet Air Time Reward 추가 (`Go2_third_test`)

- **추가 Reward**:
  - `healthy` (1.0) — 로봇이 넘어지지 않고 살아있을 때 지속적으로 보상
  - `feet_air_time` (0.3) — 발이 공중에 충분히 떠있는 trot gait 유도
- penalty 계수도 일부 조정 (`vertical_vel` 0.8, `xy_angular_vel` 0.15)
- 발을 더 명확하게 들어올리는 안정적인 trot 걸음

In [ ]:
model_name = "Go2_third_test"
env_cfg_path = f"{repo_dir}/models/{model_name}/envs.yaml"

show_code(env_cfg_path, title=model_name + " envs.yaml", max_height=600)

### 실험 3 : Hip Spread Penalty 추가 (`Go2_forth_test`)

- **추가 Reward**:
  - `hip_spread` (0.2) — 고관절(abad) 관절이 기본 자세에서 벗어나는 것에 패널티
- 실험 2(`Go2_third_test`)와의 유일한 차이: hip 관절이 안쪽으로 모이며 더 자연스러운 자세 유지
- Go2 최종 reward 설계

In [ ]:
model_name = "Go2_forth_test"
env_cfg_path = f"{repo_dir}/models/{model_name}/envs.yaml"

show_code(env_cfg_path, title=model_name + " envs.yaml", max_height=600)

---
## 3. Training

위 실험 셀에서 `model_name`과 `env_cfg_path`를 설정한 뒤, 아래 셀을 실행하면 해당 reward 설정으로 처음부터 학습합니다.

> 실습 시간에는 학습을 스킵하고 미리 학습된 모델로 Evaluation을 바로 진행합니다.

In [ ]:
train_run(env_cfg_path=env_cfg_path)

---
## 4. Evaluation

위 실험 셀에서 선택한 `model_name`의 학습된 정책으로 롤아웃을 생성합니다.

In [ ]:
video_path = eval_run()

In [ ]:
from IPython.display import Video, display

print(f"model: {model_name}")
display(
    Video(
        video_path,
        embed=True,
        html_attributes="controls autoplay loop"
    )
)

---

## 5. 결과 비교

각 실험의 Evaluation 영상을 나란히 비교합니다.

실험 1~4를 모두 실행한 후 아래 셀을 실행하세요.

In [ ]:
import base64
from IPython.display import HTML
from pathlib import Path

def video_embed(path, caption, width=320, autoplay=True):
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("ascii")
    attrs = "controls"
    if autoplay:
        attrs += " autoplay muted loop"
    return f"""
    <figure style="margin:0; text-align:center;">
      <video {attrs} width="{width}">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      </video>
      <figcaption style="margin-top:6px; font-size:12px;">{caption}</figcaption>
    </figure>
    """

videos = [
    (
        f"{repo_dir}/../rollout_Go2_second_test.mp4",
        "실험 1: + Regularization + Gait + Calf Termination",
    ),
    (
        f"{repo_dir}/../rollout_Go2_third_test.mp4",
        "실험 2: + Healthy + Feet Air Time",
    ),
    (
        f"{repo_dir}/../rollout_Go2_forth_test.mp4",
        "실험 3: + Hip Spread Penalty",
    ),
]

HTML(f"""
<div style="display:flex; gap:12px; flex-wrap:wrap;">
  {''.join([video_embed(p, cap, width=300) for p, cap in videos if Path(p).exists()])}
</div>
""")